# Micro-Ablation Study (ORACLE ONLY): S4b + S7a — CURRENT Logic

This notebook runs **only** the oracle variants **S4b** and **S7a** and saves **exactly two outputs**:
- `S4b_rrf_best_ALL.csv`
- `S7a_rerank_best_ALL.csv`

Baselines are taken from the main ablation run and are **not** executed here.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ==========================================================
# 0) CONFIG — ORACLE ONLY (S4b + S7a)
# ==========================================================
import os, re, math, json
from typing import Any, Dict, List, Tuple, Optional, Iterable
from collections import defaultdict
import numpy as np
import pandas as pd

pd.options.display.float_format = '{:.4f}'.format

# --- Paths (edit as needed) ---
REV5_CATALOG_PATH = r"/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl"
REV4_CATALOG_PATH = r"/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl"

REV5_GOLD_CSV = r"/content/drive/MyDrive/compliance_data/gold_standard_datasets/nist800_53/nist_sp800-53_rev5_gold-set_100q.csv"
REV4_GOLD_CSV = r"/content/drive/MyDrive/compliance_data/gold_standard_datasets/nist800_53/nist_sp800-53_rev4_gold-set_36q.csv"
ERROR_BANK_CSV = r"/content/drive/MyDrive/compliance_data/error_bank/error_bank_v1.csv"

QUR_CSV_REV5 = r"/content/drive/MyDrive/compliance_outputs/outputs_qur/qur_rewrites_rev5.csv"
QUR_CSV_REV4 = r"/content/drive/MyDrive/compliance_outputs/outputs_qur/qur_rewrites_rev4.csv"
QUR_CSV_ERRB = r"/content/drive/MyDrive/compliance_outputs/outputs_qur/qur_rewrites_error_bank.csv"

ABLATION_OUTPUT_DIR = r"/content/drive/MyDrive/ablation_outputs/micro_ablations"
BASE_OUT_DIR = ABLATION_OUTPUT_DIR
os.makedirs(BASE_OUT_DIR, exist_ok=True)

# --- Models ---
DENSE_MODEL_ID = "intfloat/e5-base-v2"
RERANKER_MODEL_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# --- Core retrieval hyperparams ---
CANDIDATE_SET_SIZE = 50
RRF_K_DEFAULT = 60

REWRITE_WEIGHT = 0.25      # weight for rewrites in weighted RRF fusion
REWRITE_JACCARD_MIN = 0.15 # rewrite filter threshold used by get_qur_variants_filtered
RERANK_ALPHA = 0.65        # safe blending weight for (base_rrf vs reranker)

# --- Micro settings ---
TOP_N = 10
BM25_K1 = 1.5
BM25_B = 0.75  # matches AblationStudy_S1_8.ipynb
NUM_REWRITES_FOR_SELECTION = 3  # requirement: "generate all 3 rewrites"

# --- Rerank no-harm gate threshold (rerank no-harm gate threshold) ---
RERANK_APPLY_MIN_MARGIN_RATIO = 0.15

print("[INFO] Config loaded.")


[INFO] Config loaded.


In [3]:
# ==========================================================
# 1) TEXT + CONTROL-ID NORMALIZATION & JSONL LOADERS
# ==========================================================
PUNCT_RE = re.compile(r"[^0-9A-Za-z_\s]+")
WS_RE = re.compile(r"\s+")
TOKEN_RE = re.compile(r"[A-Za-z0-9]+")

def normalize_text(text: str) -> str:
    if not isinstance(text, str): text = "" if pd.isna(text) else str(text)
    s = text.lower().replace('-', ' ')
    s = PUNCT_RE.sub(' ', s)
    return WS_RE.sub(' ', s).strip()

def tokenize(text: str) -> List[str]:
    return [t.lower() for t in TOKEN_RE.findall(str(text))]

def normalize_control_id(control_id: str) -> str:
    if not isinstance(control_id, str): return ""
    return control_id.strip().upper().replace("_", "-")

def control_id_aliases(control_id: str) -> str:
    cid = normalize_control_id(control_id)
    return f"{cid} {cid.replace('-', '')} {cid.replace('-', ' ')}"

def load_clause_records_jsonl(jsonl_path: str, keep_kinds=("smt", "gdn")) -> List[Dict]:
    out = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            rec = json.loads(line)
            if keep_kinds and str(rec.get("kind","")).lower() not in keep_kinds: continue

            txt = str(rec.get("text","")).strip()
            if not txt: continue

            out.append({
                "clause_id": str(rec.get("id","")).strip(),
                "control_id": normalize_control_id(rec.get("control_id") or rec.get("control") or ""),
                "title": str(rec.get("title","")).strip(),
                "text": txt,
                "kind": str(rec.get("kind","")).lower()
            })
    return out

def build_control_docs_from_clauses(records):
    by_ctl = defaultdict(list)
    titles = {}
    for r in records:
        cid = r["control_id"]
        if not cid: continue
        by_ctl[cid].append(r["text"])
        if r["title"]: titles[cid] = r["title"]

    docs = []
    for cid, parts in by_ctl.items():
        header = [control_id_aliases(cid)]
        if titles.get(cid): header.append(titles[cid])
        full_text = "\n".join(header + parts)
        docs.append({"control_id": cid, "text": full_text})
    return docs

def build_control_fallback_text_map(records):
    # For reranker fallback
    by_ctl = defaultdict(list)
    for r in records:
        cid = r["control_id"]
        if cid: by_ctl[cid].append(r["text"])
    return {k: "\n".join(v[:8]) for k,v in by_ctl.items()}

In [4]:
# ==========================================================
# 2) BM25 + METRICS
# ==========================================================
class BM25Okapi:
    def __init__(self, corpus_tokens, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.N = len(corpus_tokens)
        self.doc_len = np.array([len(d) for d in corpus_tokens], dtype=np.float32)
        self.avgdl = float(self.doc_len.mean()) if self.N else 0.0

        self.tf = []
        df = {}
        for doc in corpus_tokens:
            tf_doc = {}
            for t in doc:
                tf_doc[t] = tf_doc.get(t, 0) + 1
            self.tf.append(tf_doc)
            for t in tf_doc:
                df[t] = df.get(t, 0) + 1

        self.idf = {t: math.log((self.N - freq + 0.5)/(freq + 0.5) + 1) for t, freq in df.items()}

    def get_scores(self, query_tokens):
        scores = np.zeros(self.N, dtype=np.float32)
        for t in query_tokens:
            if t not in self.idf: continue
            idf = self.idf[t]
            for i, tf_doc in enumerate(self.tf):
                f = tf_doc.get(t, 0)
                if f:
                    num = f * (self.k1 + 1)
                    den = f + self.k1 * (1 - self.b + self.b * (self.doc_len[i]/self.avgdl))
                    scores[i] += idf * (num/den)
        return scores

    def get_top_n(self, query_tokens, n=10):
        scores = self.get_scores(query_tokens)
        if n >= len(scores): return np.argsort(-scores).tolist()
        idx = np.argpartition(-scores, n-1)[:n]
        return idx[np.argsort(-scores[idx])].tolist()

def build_bm25(docs):
    assert 'BM25_K1' in globals() and 'BM25_B' in globals(), 'BM25_K1/BM25_B not set in config'

    ids = [d["control_id"] for d in docs]
    toks = [tokenize(normalize_text(d["text"])) for d in docs]
    return BM25Okapi(toks, k1=BM25_K1, b=BM25_B), ids


In [5]:
# ==========================================================
# 3) DENSE RETRIEVER SETUP
# ==========================================================
try:
    import faiss
    import sentence_transformers
except:
    !pip install -q sentence-transformers faiss-cpu

import torch
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

class DenseIndex:
    def __init__(self, model_id):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Loading Dense Model {model_id} on {self.device}...")
        self.model = SentenceTransformer(model_id, device=self.device)
        self.index = None
        self.ids = []

    def build(self, texts, ids):
        self.ids = ids
        emb = self.model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(emb)

    def search(self, query, top_k):
        q_emb = self.model.encode([query], normalize_embeddings=True)
        scores, idxs = self.index.search(q_emb, top_k)
        return [(self.ids[i], float(s)) for i, s in zip(idxs[0], scores[0]) if i != -1]

# Clause-to-Control Adapter
class ClauseDenseControlAdapter:
    def __init__(self, dense_idx, clause_map, text_map):
        self.idx = dense_idx
        self.cmap = clause_map
        self.tmap = text_map
        self.last_best_clause = {} # query-specific cache

    def search(self, query, top_k=50):
        # Search more clauses to ensure we get enough unique controls
        hits = self.idx.search(query, top_k=top_k*5)
        best_score = {}
        best_clause = {}

        for cid, score in hits:
            ctl = self.cmap.get(cid)
            if not ctl: continue
            ctl = normalize_control_id(ctl)
            if score > best_score.get(ctl, -1.0):
                best_score[ctl] = score
                best_clause[ctl] = cid

        self.last_best_clause = best_clause
        ranked = sorted(best_score.items(), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

    def best_clause_text_for_control(self, ctl_id):
        cid = normalize_control_id(ctl_id)
        clid = self.last_best_clause.get(cid)
        return self.tmap.get(clid) if clid else None

def build_dense_retriever(jsonl_path):
    recs = load_clause_records_jsonl(jsonl_path, keep_kinds={"smt", "gdn"})
    ids, texts = [], []
    cmap, tmap = {}, {}
    for r in recs:
        header = f"{r['control_id']} {r['title']} ({r['kind']})"
        full = f"{header}\n{r['text']}"
        ids.append(r["clause_id"])
        texts.append(f"passage: {full}") # e5 prefix
        cmap[r["clause_id"]] = r["control_id"]
        tmap[r["clause_id"]] = full

    idx = DenseIndex(DENSE_MODEL_ID)
    idx.build(texts, ids)
    return ClauseDenseControlAdapter(idx, cmap, tmap)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 18.5 MB/s eta 0:00:00


In [6]:
# ==========================================================
# 1) LOAD DATA & BUILD INDICES (Control-level BM25, Clause-level Dense Adapter)
# ==========================================================
print("--- Loading Gold Sets ---")
rev5_gold = pd.read_csv(REV5_GOLD_CSV, encoding='ISO-8859-1')
rev4_gold = pd.read_csv(REV4_GOLD_CSV, encoding='ISO-8859-1')

print("--- Loading Error Bank ---")
error_bank_df = pd.read_csv(ERROR_BANK_CSV, encoding='ISO-8859-1')

print("--- Loading QUR rewrites ---")
qur_rev5_df = pd.read_csv(QUR_CSV_REV5, encoding='ISO-8859-1')
qur_rev4_df = pd.read_csv(QUR_CSV_REV4, encoding='ISO-8859-1')
qur_errb_df = pd.read_csv(QUR_CSV_ERRB, encoding='ISO-8859-1')

print("--- Building BM25 Indices (Control-level docs built from clause CCS) ---")
rev5_clause_recs = load_clause_records_jsonl(REV5_CATALOG_PATH)
rev4_clause_recs = load_clause_records_jsonl(REV4_CATALOG_PATH)

# --- SAFETY CHECK: ensure we loaded the FLAT clause-level CCS (not OSCAL nested / not empty) ---
print('--- CCS loader sanity check ---')
print('  Rev5 clause recs:', len(rev5_clause_recs), ' Rev4 clause recs:', len(rev4_clause_recs))
assert len(rev5_clause_recs) > 1000 and len(rev4_clause_recs) > 1000, 'Loader returned too few clause records; check JSONL path/schema.'
assert any('_smt' in str(r.get('clause_id','')) for r in rev5_clause_recs[:500]), 'Expected clause IDs containing _smt; check you are using clause-level CCS.'
print('  Sample rev5 record keys:', list(rev5_clause_recs[0].keys()))
print('  Sample rev5 record:', {k: rev5_clause_recs[0][k] for k in ['clause_id','control_id','kind'] if k in rev5_clause_recs[0]})


bm25_rev5, rev5_control_ids = build_bm25(build_control_docs_from_clauses(rev5_clause_recs))
bm25_rev4, rev4_control_ids = build_bm25(build_control_docs_from_clauses(rev4_clause_recs))

print("--- Building Dense Indices (Clause-level → Control-level adapter) ---")
dense_rev5 = build_dense_retriever(REV5_CATALOG_PATH)
dense_rev4 = build_dense_retriever(REV4_CATALOG_PATH)

print("--- Loading Reranker ---")
import torch
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder(RERANKER_MODEL_ID, device='cuda' if torch.cuda.is_available() else 'cpu')

# Fallback control→text maps (used if dense doesn't provide best clause text)
rev5_control_to_text = build_control_fallback_text_map(rev5_clause_recs)
rev4_control_to_text = build_control_fallback_text_map(rev4_clause_recs)

print("[INFO] System ready.")
print("  Rev5 controls:", len(rev5_control_ids), " Rev4 controls:", len(rev4_control_ids))

--- Loading Gold Sets ---
--- Loading Error Bank ---
--- Loading QUR rewrites ---
--- Building BM25 Indices (Control-level docs built from clause CCS) ---
--- CCS loader sanity check ---
  Rev5 clause recs: 3136  Rev4 clause recs: 2281
  Sample rev5 record keys: ['clause_id', 'control_id', 'title', 'text', 'kind']
  Sample rev5 record: {'clause_id': 'ac-1_smt', 'control_id': 'AC-1', 'kind': 'smt'}
--- Building Dense Indices (Clause-level → Control-level adapter) ---
Loading Dense Model intfloat/e5-base-v2 on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

[INFO] System ready.
  Rev5 controls: 1013  Rev4 controls: 826


In [7]:
# ==========================================================
# 2) SYSTEM LOGIC — ONLY S4b + S7a (+ required helpers)
# ==========================================================
import numpy as np
import pandas as pd
import json

def find_rank(gold, ranked_list, top_n):
    g = normalize_control_id(gold)
    for i, c in enumerate(ranked_list[:top_n], 1):
        if normalize_control_id(c) == g:
            return i
    return 0
def strict_clean(s: str) -> str:
    """Strict alnum-only key for stable exact matching."""
    return "".join(c.lower() for c in str(s) if c.isalnum())
def jaccard_overlap(a: str, b: str) -> float:
    """Token Jaccard overlap between two queries (0..1)."""
    A = set(tokenize(normalize_text(a)))
    B = set(tokenize(normalize_text(b)))
    if not A or not B: return 0.0
    return len(A & B) / float(len(A | B))
def normalize_scores_minmax(x: np.ndarray) -> np.ndarray:
    """Min-max normalize to [0,1]. Returns zeros if constant."""
    if x.size == 0: return x
    mn, mx = float(np.min(x)), float(np.max(x))
    if mx - mn < 1e-12: return np.zeros_like(x, dtype=np.float32)
    return ((x - mn) / (mx - mn)).astype(np.float32)
def get_qur_variants_filtered(original_query: str, qur_df: pd.DataFrame,
                              max_rewrites: int = 3, jaccard_min: float = 0.15) -> list:
    """Return [original] + filtered rewrites (deduped)."""
    variants = [original_query]
    if qur_df is None or qur_df.empty: return variants

    if 'strict_key' not in qur_df.columns and 'original_query' in qur_df.columns:
        qur_df['strict_key'] = qur_df['original_query'].apply(strict_clean)

    q_key = strict_clean(original_query)
    matches = qur_df[qur_df['strict_key'] == q_key] if 'strict_key' in qur_df.columns else pd.DataFrame()

    if matches.empty: return variants

    rewrites = []
    for r in matches['rewritten_query'].tolist():
        r = str(r).strip()
        if len(r) < 6: continue
        if jaccard_overlap(original_query, r) < jaccard_min: continue
        rewrites.append(r)

    variants.extend(rewrites[:max_rewrites])
    return list(dict.fromkeys(variants)) # stable dedupe
def rrf_fuse(lists_of_cids, k=60, weights=None):
    """Reciprocal Rank Fusion with optional list-weights."""
    scores = defaultdict(float)
    if weights is None: weights = [1.0] * len(lists_of_cids)
    for w, lst in zip(weights, lists_of_cids):
        for r, cid in enumerate(lst, 1):
            scores[cid] += float(w) * (1.0 / (k + r))
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# --- Core Logic Helpers (The "Brains") ---
def s7_heavy_retrieve_details(
    original_query,
    bm25_retriever,
    dense_retriever,
    bm25_control_ids,
    cross_enc,
    id_to_text_map,
    qur_df,
    candidate_set_size=50,
    rrf_k=60,
    max_rewrites=3,
    rewrite_weight=0.25,
    rewrite_jaccard_min=0.15,
    rerank_alpha=0.65,
    rerank_query_override=None,
    rerank_apply_min_margin_ratio=0.15,   # NEW: no-harm gate threshold
):
    """
    Hybrid retrieval: Weighted RRF + Safe Rerank Blending + no-harm gate.

    No-Harm Gate:
      - If reranker changes top1 but its margin is small, keep base ordering.
      - Prevents 'rank reversal' on ambiguous candidates.
    """

    variants = get_qur_variants_filtered(original_query, qur_df, max_rewrites, rewrite_jaccard_min)
    rerank_query = rerank_query_override or original_query
    weights = [1.0] + [rewrite_weight] * (len(variants) - 1)

    # 1) Weighted RRF retrieval
    rrf_scores = defaultdict(float)
    best_text_cache = {}

    for q, w in zip(variants, weights):
        # BM25
        q_toks = tokenize(normalize_text(q))
        idxs = bm25_retriever.get_top_n(q_toks, n=candidate_set_size)
        for r, i in enumerate(idxs, 1):
            cid = normalize_control_id(bm25_control_ids[i]).upper()
            rrf_scores[cid] += float(w) * (1.0 / (rrf_k + r))
            if cid not in best_text_cache:
                best_text_cache[cid] = id_to_text_map.get(cid, "")

        # Dense
        try:
            hits = dense_retriever.search(q, top_k=candidate_set_size)
        except Exception:
            hits = []

        for r, (raw_cid, _) in enumerate(hits, 1):
            cid = normalize_control_id(str(raw_cid)).upper()
            rrf_scores[cid] += float(w) * (1.0 / (rrf_k + r))
            if hasattr(dense_retriever, "best_clause_text_for_control"):
                t = dense_retriever.best_clause_text_for_control(cid)
                if t:
                    best_text_cache[cid] = t

    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:candidate_set_size]
    cids_rerank = [c for c, _ in fused]

    if not cids_rerank:
        return {"final_ranked_cids": [], "meta": {"note": "no_candidates", "rerank_applied": False}}

    # 2) Build cross-encoder pairs
    pairs = []
    valid_cids = []
    for cid in cids_rerank:
        txt = best_text_cache.get(cid) or id_to_text_map.get(cid, "")
        if txt:
            pairs.append([rerank_query, txt])
            valid_cids.append(cid)

    base_ranked_cids = valid_cids[:]  # base order = weighted RRF order

    # If we have no text, fallback to base order
    if not pairs:
        return {
            "final_ranked_cids": cids_rerank,
            "meta": {
                "note": "no_text_for_rerank",
                "rerank_applied": False,
                "num_variants": len(variants),
                "base_top1": cids_rerank[0] if cids_rerank else "",
            }
        }

    # 3) Cross-encoder scoring
    pred = cross_enc.predict(pairs, show_progress_bar=False)

    # Base scores (RRF) aligned to valid_cids
    base = np.array([rrf_scores.get(c, 0.0) for c in valid_cids], dtype=np.float32)
    base_n = normalize_scores_minmax(base)
    pred_n = normalize_scores_minmax(np.array(pred, dtype=np.float32))

    # Safe blending score
    final_scores = float(rerank_alpha) * base_n + (1.0 - float(rerank_alpha)) * pred_n

    # 4) Reranked order by blended score
    rerank_order = np.argsort(-final_scores)
    reranked_cids = [valid_cids[i] for i in rerank_order]

    # Compute rerank margin ratio (top1 vs top2)
    if len(rerank_order) >= 2:
        fs1 = float(final_scores[rerank_order[0]])
        fs2 = float(final_scores[rerank_order[1]])
        rerank_margin_ratio = (fs1 - fs2) / max(fs1, 1e-9)
    else:
        rerank_margin_ratio = 1.0

    # Compute base margin ratio (top1 vs top2) using base signal
    if len(base_n) >= 2:
        bs1 = float(base_n[0])
        bs2 = float(base_n[1])
        base_margin_ratio = (bs1 - bs2) / max(bs1, 1e-9)
    else:
        base_margin_ratio = 1.0

    base_top1 = base_ranked_cids[0] if base_ranked_cids else ""
    rerank_top1 = reranked_cids[0] if reranked_cids else ""

    # 5) No-Harm Rerank Gate:
    # If reranker flips top1 but has low margin, keep base order.
    rerank_applied = True
    final_ranked_cids = reranked_cids

    if (rerank_top1 != base_top1) and (rerank_margin_ratio < float(rerank_apply_min_margin_ratio)):
        rerank_applied = False
        final_ranked_cids = base_ranked_cids

    # Final margin depends on which ordering we used
    final_margin_ratio = rerank_margin_ratio if rerank_applied else base_margin_ratio
    final_top1 = final_ranked_cids[0] if final_ranked_cids else ""

    return {
        "final_ranked_cids": final_ranked_cids,
        "meta": {
            "num_variants": len(variants),
            "rerank_applied": rerank_applied,
            "rerank_alpha": float(rerank_alpha),
            "rerank_apply_min_margin_ratio": float(rerank_apply_min_margin_ratio),
            "base_top1": base_top1,
            "rerank_top1": rerank_top1,
            "final_top1": final_top1,
            "base_margin_ratio": float(base_margin_ratio),
            "rerank_margin_ratio": float(rerank_margin_ratio),
            "final_margin_ratio": float(final_margin_ratio),
        }
    }

    lists = []
    for v in variants:
        q_toks = tokenize(normalize_text(v))
        idxs = bm25.get_top_n(q_toks, n=top_n)
        lists.append([normalize_control_id(doc_ids[i]) for i in idxs])

    fused = rrf_fuse(lists, k=rrf_k, weights=weights)
    retrieved = [c for c, _ in fused][:top_n]
    return {"rank": find_rank(gold, retrieved, top_n), "retrieved_ids": retrieved, "system_meta": {"sys": "S4b", "n_vars": len(variants)}}
# ==========================================================
# 2) MICRO-ABlation Helpers (Best Rewrite Selection + Variants)
# ==========================================================

def get_top_k_rewrites(original_query: str, qur_df: pd.DataFrame, k: int = 3) -> List[str]:
    """Return up to k rewrites for original_query, ordered by rewrite_rank asc."""
    if qur_df is None or qur_df.empty:
        return []
    q_key = strict_clean(original_query)
    if 'strict_key' not in qur_df.columns:
        qur_df['strict_key'] = qur_df['original_query'].apply(strict_clean)
    m = qur_df[qur_df['strict_key'] == q_key]
    if m.empty:
        return []
    if 'rewrite_rank' in m.columns:
        m = m.sort_values('rewrite_rank', ascending=True)
    rewrites = [str(x).strip() for x in m['rewritten_query'].tolist() if str(x).strip()]
    out = []
    for r in rewrites:
        if r not in out:
            out.append(r)
        if len(out) >= k:
            break
    return out

def stage1_hybrid_rrf_ranked(
    query: str,
    bm25_retriever,
    dense_retriever,
    bm25_control_ids: List[str],
    candidate_set_size: int = 50,
    rrf_k: int = 60,
) -> Tuple[List[str], Dict[str, float]]:
    """Stage-1 retrieval: BM25 + Dense → RRF fuse, returning ranked controls and score map."""
    q_toks = tokenize(normalize_text(query))
    idxs = bm25_retriever.get_top_n(q_toks, n=candidate_set_size)
    bm25_list = [normalize_control_id(bm25_control_ids[i]).upper() for i in idxs]

    try:
        dense_hits = dense_retriever.search(query, top_k=candidate_set_size)
    except Exception:
        dense_hits = []
    dense_list = [normalize_control_id(str(cid)).upper() for cid, _ in dense_hits]

    fused = rrf_fuse([bm25_list, dense_list], k=rrf_k, weights=[1.0, 1.0])
    ranked = [cid for cid, _ in fused][:candidate_set_size]
    score_map = {cid: float(score) for cid, score in fused}
    return ranked, score_map

def choose_best_rewrite_for_gold(
    original_query: str,
    gold_control_id: str,
    bm25_retriever,
    dense_retriever,
    bm25_control_ids: List[str],
    qur_df: pd.DataFrame,
    candidate_set_size: int = 50,
    rrf_k: int = 60,
    k_rewrites: int = 3,
) -> Optional[str]:
    """Requirement B: pick rewrite that gives best (lowest) Stage-1 fused rank of gold control."""
    rewrites = get_top_k_rewrites(original_query, qur_df, k=k_rewrites)
    if not rewrites:
        return None

    gold = normalize_control_id(gold_control_id).upper()

    best = None
    best_rank = 10**9
    best_tiebreak = -1.0

    for r in rewrites:
        ranked, score_map = stage1_hybrid_rrf_ranked(
            r, bm25_retriever, dense_retriever, bm25_control_ids,
            candidate_set_size=candidate_set_size, rrf_k=rrf_k
        )

        rnk = 10**9
        for i, cid in enumerate(ranked, 1):
            if cid == gold:
                rnk = i
                break

        tb = float(score_map.get(gold, -1.0)) if gold in score_map else (float(score_map.get(ranked[0], -1.0)) if ranked else -1.0)

        if (rnk < best_rank) or (rnk == best_rank and tb > best_tiebreak):
            best = r
            best_rank = rnk
            best_tiebreak = tb

    return best

def system_s4b_rrf_best(q, gold, bm25, doc_ids, qur_df, top_n=10, rrf_k=60,
                       bm25_for_selection=None, dense_for_selection=None, ids_for_selection=None):
    """S4b: Original + best-scoring rewrite only (weighted RRF over BM25)."""
    best_rewrite = None
    if bm25_for_selection is not None and dense_for_selection is not None and ids_for_selection is not None:
        best_rewrite = choose_best_rewrite_for_gold(
            q, gold, bm25_for_selection, dense_for_selection, ids_for_selection, qur_df,
            candidate_set_size=CANDIDATE_SET_SIZE, rrf_k=RRF_K_DEFAULT, k_rewrites=NUM_REWRITES_FOR_SELECTION
        )

    variants = [q] + ([best_rewrite] if best_rewrite else [])
    weights = [1.0] + [REWRITE_WEIGHT] * (len(variants) - 1)

    lists = []
    for v in variants:
        q_toks = tokenize(normalize_text(v))
        idxs = bm25.get_top_n(q_toks, n=top_n)
        lists.append([normalize_control_id(doc_ids[i]) for i in idxs])

    fused = rrf_fuse(lists, k=rrf_k, weights=weights)
    retrieved = [c for c, _ in fused][:top_n]
    return {"rank": find_rank(gold, retrieved, top_n),
            "retrieved_ids": retrieved,
            "system_meta": {"sys": "S4b", "n_vars": len(variants), "best_rewrite_used": bool(best_rewrite)}}

def system_s7a_rerank_best(q, gold, bm25, doc_ids, dense, id_to_text_map, qur_df, top_n=10,
                          bm25_for_selection=None, dense_for_selection=None, ids_for_selection=None):
    """S7a: Candidate generation unchanged; cross-encoder pairs use best-scoring rewrite."""
    best_rewrite = None
    if bm25_for_selection is not None and dense_for_selection is not None and ids_for_selection is not None:
        best_rewrite = choose_best_rewrite_for_gold(
            q, gold, bm25_for_selection, dense_for_selection, ids_for_selection, qur_df,
            candidate_set_size=CANDIDATE_SET_SIZE, rrf_k=RRF_K_DEFAULT, k_rewrites=NUM_REWRITES_FOR_SELECTION
        )

    heavy = s7_heavy_retrieve_details(
        q, bm25, dense, doc_ids, cross_encoder, id_to_text_map, qur_df,
        candidate_set_size=CANDIDATE_SET_SIZE, rrf_k=RRF_K_DEFAULT,
        rewrite_weight=REWRITE_WEIGHT, rewrite_jaccard_min=REWRITE_JACCARD_MIN, rerank_alpha=RERANK_ALPHA,
        rerank_apply_min_margin_ratio=RERANK_APPLY_MIN_MARGIN_RATIO,
        rerank_query_override=best_rewrite
    )
    final = heavy["final_ranked_cids"][:top_n]
    meta = heavy.get("meta", {})
    meta["sys"] = "S7a"
    meta["best_rewrite_used"] = bool(best_rewrite)
    return {"rank": find_rank(gold, final, top_n), "retrieved_ids": final, "system_meta": meta}


In [8]:
# ==========================================================
# 3) EVALUATION (Control-level hit)
# ==========================================================
def _hit_at_k(gold_cid: str, retrieved: List[str], k: int) -> bool:
    g = normalize_control_id(gold_cid).upper()
    for cid in retrieved[:k]:
        if normalize_control_id(cid).upper() == g:
            return True
    return False

def _rank_of_gold(gold_cid: str, retrieved: List[str], k: int) -> Optional[int]:
    g = normalize_control_id(gold_cid).upper()
    for i, cid in enumerate(retrieved[:k], 1):
        if normalize_control_id(cid).upper() == g:
            return i
    return None

def _mrr_at_k(gold_cid: str, retrieved: List[str], k: int) -> float:
    r = _rank_of_gold(gold_cid, retrieved, k)
    return 0.0 if r is None else 1.0 / float(r)

def _ndcg_at_k_single(gold_cid: str, retrieved: List[str], k: int) -> float:
    r = _rank_of_gold(gold_cid, retrieved, k)
    if r is None:
        return 0.0
    return 1.0 / math.log2(r + 1)

def evaluate_dataset(
    tag: str,
    df: pd.DataFrame,
    run_fn,
    *,
    gold_col: str,
    question_col: str = "question",
    dataset_tag: Optional[str] = None,
) -> pd.DataFrame:
    """
    Run a system over a dataset and return a per-query results DataFrame.
    Run a system over a dataset and return a per-query results DataFrame.
    """
    rows = []
    for idx, row in df.iterrows():
        q = str(row[question_col])
        gold = str(row[gold_col])

        res = run_fn(q, gold)
        retrieved = res.get("retrieved_ids", []) or []
        meta = res.get("system_meta", {}) or {}

        rank10 = _rank_of_gold(gold, retrieved, 10)
        rows.append({
            "dataset": dataset_tag or "",
            "question_id": row.get("ID", row.get("id", row.get("question_id", idx))),
            "question": q,
            "gold_control_id": normalize_control_id(gold),
            "rank": rank10 if rank10 is not None else "",
            "is_hit_at_1": _hit_at_k(gold, retrieved, 1),
            "is_hit_at_5": _hit_at_k(gold, retrieved, 5),
            "is_hit_at_10": _hit_at_k(gold, retrieved, 10),
            "mrr@10": _mrr_at_k(gold, retrieved, 10),
            "ndcg@10": _ndcg_at_k_single(gold, retrieved, 10),
            "top_1_hit": normalize_control_id(retrieved[0]) if retrieved else "",
            "retrieved_control_ids": " | ".join([normalize_control_id(x) for x in retrieved[:10]]),
            "system_meta": json.dumps(meta, ensure_ascii=False),
        })

    out_df = pd.DataFrame(rows)

    return out_df


In [9]:
# ==========================================================
# 4) RUN (ORACLE ONLY): S4b + S7a — WRITE EXACTLY TWO CSVs
#     Output policy: ONLY TWO FILES total:
#       - S4b_rrf_best_ALL.csv
#       - S7a_rerank_best_ALL.csv
#     Baselines are taken from the main ablation notebook outputs.
# ==========================================================

# --- SAFETY CHECK: make sure prerequisites were executed (Kernel -> Restart & Run All recommended) ---
missing = [name for name in ['bm25_rev5','bm25_rev4','dense_rev5','dense_rev4','cross_encoder','rev5_control_ids','rev4_control_ids'] if name not in globals()]
assert not missing, f"Missing objects: {missing}. Run the setup cells above."
assert callable(globals().get('choose_best_rewrite_for_gold', None)), "choose_best_rewrite_for_gold is not defined; run the system logic cell."

# --- Split Error Bank by version ---
_ver_col = "version" if "version" in error_bank_df.columns else ("framework_version" if "framework_version" in error_bank_df.columns else None)
assert _ver_col is not None, "Error Bank must contain a 'version' (or 'framework_version') column."

err_rev5 = error_bank_df[error_bank_df[_ver_col].astype(str).str.contains("rev5", case=False, na=False)].copy()
err_rev4 = error_bank_df[error_bank_df[_ver_col].astype(str).str.contains("rev4", case=False, na=False)].copy()

# --- Dataset plan (dataset_tag, df, bm25, control_ids, dense, control_to_text, qur_df, gold_col) ---
datasets = [
    ("rev5_gold", rev5_gold, bm25_rev5, rev5_control_ids, dense_rev5, rev5_control_to_text, qur_rev5_df, "control_id"),
    ("rev4_gold", rev4_gold, bm25_rev4, rev4_control_ids, dense_rev4, rev4_control_to_text, qur_rev4_df, "control_id"),
    ("error_bank_rev5", err_rev5, bm25_rev5, rev5_control_ids, dense_rev5, rev5_control_to_text, qur_errb_df, "gold_control_id"),
    ("error_bank_rev4", err_rev4, bm25_rev4, rev4_control_ids, dense_rev4, rev4_control_to_text, qur_errb_df, "gold_control_id"),
]

def _summ(df: pd.DataFrame) -> Dict[str, float]:
    if df is None or len(df) == 0:
        return {"Recall@1": 0.0, "Recall@5": 0.0, "Recall@10": 0.0, "MRR@10": 0.0, "nDCG@10": 0.0}
    return {
        "Recall@1": float(df["is_hit_at_1"].mean()),
        "Recall@5": float(df["is_hit_at_5"].mean()),
        "Recall@10": float(df["is_hit_at_10"].mean()),
        "MRR@10": float(df["mrr@10"].mean()),
        "nDCG@10": float(df["ndcg@10"].mean()),
    }

s4b_all = []
s7a_all = []
summary_rows = []

for (ds_tag, df, bm25, control_ids, dense, ctl_to_text, qur_df, gold_col) in datasets:
    if df is None or len(df) == 0:
        print(f"[SKIP] {ds_tag}: empty dataset")
        continue

    print(f"\n=== Running ORACLE variants on: {ds_tag} (N={len(df)}) ===")

    # --- S4b (Oracle best rewrite for fusion breadth test) ---
    s4b_df = evaluate_dataset(
        f"S4b_rrf_best ({ds_tag})",
        df,
        lambda q,g: system_s4b_rrf_best(
            q, g,
            bm25, control_ids, qur_df,
            top_n=TOP_N, rrf_k=RRF_K_DEFAULT,
            bm25_for_selection=bm25, dense_for_selection=dense, ids_for_selection=control_ids
        ),
        gold_col=gold_col,
        dataset_tag=ds_tag,
    )
    s4b_all.append(s4b_df)

    # --- S7a (Oracle best rewrite for rerank query source test) ---
    s7a_df = evaluate_dataset(
        f"S7a_rerank_best ({ds_tag})",
        df,
        lambda q,g: system_s7a_rerank_best(
            q, g,
            bm25, control_ids, dense, ctl_to_text, qur_df,
            top_n=TOP_N,
            bm25_for_selection=bm25, dense_for_selection=dense, ids_for_selection=control_ids
        ),
        gold_col=gold_col,
        dataset_tag=ds_tag,
    )
    s7a_all.append(s7a_df)

    # --- Console summary (not saved as extra files) ---
    s4b_m = _summ(s4b_df)
    s7a_m = _summ(s7a_df)
    summary_rows.append({"dataset": ds_tag, "system": "S4b_rrf_best", **s4b_m})
    summary_rows.append({"dataset": ds_tag, "system": "S7a_rerank_best", **s7a_m})

# --- Write ONLY TWO output files ---
assert len(s4b_all) > 0 and len(s7a_all) > 0, "No results were produced — check dataset splits and inputs."

s4b_combined = pd.concat(s4b_all, ignore_index=True)
s7a_combined = pd.concat(s7a_all, ignore_index=True)

out_s4b = os.path.join(ABLATION_OUTPUT_DIR, "S4b_rrf_best_ALL.csv")
out_s7a = os.path.join(ABLATION_OUTPUT_DIR, "S7a_rerank_best_ALL.csv")

s4b_combined.to_csv(out_s4b, index=False, encoding="utf-8")
s7a_combined.to_csv(out_s7a, index=False, encoding="utf-8")

print("\n[OK] Wrote ONLY these two files:")
print(" ", out_s4b)
print(" ", out_s7a)

# --- Print summary table (console only) ---
summary_df = pd.DataFrame(summary_rows)
if len(summary_df):
    print("\n=== Summary (console only) ===")
    display(summary_df)



=== Running ORACLE variants on: rev5_gold (N=100) ===

=== Running ORACLE variants on: rev4_gold (N=36) ===

=== Running ORACLE variants on: error_bank_rev5 (N=27) ===

=== Running ORACLE variants on: error_bank_rev4 (N=12) ===

[OK] Wrote ONLY these two files:
  /content/drive/MyDrive/ablation_outputs/micro_ablations/S4b_rrf_best_ALL.csv
  /content/drive/MyDrive/ablation_outputs/micro_ablations/S7a_rerank_best_ALL.csv

=== Summary (console only) ===


,dataset,system,Recall@1,Recall@5,Recall@10,MRR@10,nDCG@10
0,rev5_gold,S4b_rrf_best,0.7600,0.9600,0.9600,0.8482,0.8767
1,rev5_gold,S7a_rerank_best,0.8600,0.9700,0.9700,0.9075,0.9235
2,rev4_gold,S4b_rrf_best,0.7778,0.9167,0.9444,0.8356,0.8625
3,rev4_gold,S7a_rerank_best,0.8333,1.0000,1.0000,0.9028,0.9276
4,error_bank_rev5,S4b_rrf_best,0.1852,0.8519,0.8519,0.4889,0.5822
5,error_bank_rev5,S7a_rerank_best,0.5556,0.8889,0.8889,0.7099,0.7562
6,error_bank_rev4,S4b_rrf_best,0.1667,0.7500,0.8333,0.4097,0.5150
7,error_bank_rev4,S7a_rerank_best,0.4167,1.0000,1.0000,0.6806,0.7629
